### Column transformer

In [2]:
import pandas as pd
import numpy as np

In [3]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split

In [4]:
df = pd.read_csv("../data/covid_toy.csv")
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [7]:
df["city"].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [9]:
df.isna().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [6]:
X = df.drop("has_covid", axis=1)
y = df["has_covid"]

In [10]:
X_train, X_test, y_train , y_test = train_test_split(X,y,
                                                     test_size=0.2)
X_train.shape, X_test.shape

((80, 5), (20, 5))

In [11]:
X_train.head()

,age,gender,fever,cough,city
25,23,Male,NaN,Mild,Mumbai
36,38,Female,101.0,Mild,Bangalore
27,33,Female,102.0,Strong,Delhi
31,83,Male,103.0,Mild,Kolkata
76,80,Male,100.0,Mild,Bangalore


#### without column transformer

In [17]:
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[["fever"]])
X_test_fever = si.transform(X_test[["fever"]])
X_train_fever.shape

(80, 1)

In [20]:
# ordinal encoding
oe = OrdinalEncoder(categories = [["Mild", "Strong"]])
X_train_cough = oe.fit_transform(X_train[['cough']])
X_test_cough = oe.transform(X_test[['cough']])
X_train_cough.shape

(80, 1)

In [22]:
# One hot encoding
ohe = OneHotEncoder(drop="first", sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender',"city"]])
X_test_gender_city = ohe.transform(X_test[['gender',"city"]])
X_train_gender_city.shape

(80, 4)

In [24]:
# Extracting age
X_train_age = X_train.drop(["gender",'fever', 'cough','city'], axis=1).values
X_test_age = X_test.drop(["gender",'fever', 'cough','city'], axis=1).values
X_train_age.shape

(80, 1)

In [28]:
X_train_transformed = np.concatenate((X_train_age, X_train_fever,X_train_gender_city,X_train_cough), axis=1)
X_test_transformed = np.concatenate((X_test_age, X_test_fever,X_test_gender_city,X_test_cough), axis=1)
X_train_transformed.shape

(80, 7)

#### Using column transformer

In [34]:
from sklearn.compose import ColumnTransformer
transformer = ColumnTransformer(transformers =[
    ('tnf1', SimpleImputer(), ['fever']),
    ("tnf2", OrdinalEncoder(categories=[["Mild", "Strong"]]),["cough"] ),
    ("tnf3", OneHotEncoder(drop="first", sparse_output=False), ['gender', 'city'])
], remainder ="passthrough")

In [35]:
transformer.fit_transform(X_train).shape

(80, 7)

In [36]:
transformer.transform(X_test).shape

(20, 7)